# 01 - Factor-Augmented VAR (FAVAR)

This notebook covers the **Factor-Augmented VAR (FAVAR)** model, which extends the standard VAR by incorporating information from a large panel of macroeconomic indicators via latent factors.

## Topics covered

- The curse of dimensionality in monetary policy analysis
- Factor extraction via Principal Component Analysis (PCA)
- Two-step FAVAR estimation (Bernanke, Boivin & Eliasz, 2005)
- Impulse Response Functions in the FAVAR framework
- Panel-level IRFs: how policy shocks propagate to hundreds of variables
- Exercises: alternative factor specifications and robustness

---

## Motivation: Why FAVAR?

Standard VAR models are limited to a small number of variables (typically 3-8). Central banks, however, monitor **hundreds** of indicators — employment across sectors, industrial production by category, financial spreads, surveys, etc.

The FAVAR addresses this by:

1. **Extracting latent factors** $F_t$ from a large panel $X_t$ via PCA
2. **Augmenting** the VAR with these factors alongside policy variables $Y_t$

The model:

$$X_t = \Lambda^f F_t + \Lambda^y Y_t + e_t$$

$$\begin{bmatrix} F_t \\ Y_t \end{bmatrix} = \Phi(L) \begin{bmatrix} F_{t-1} \\ Y_{t-1} \end{bmatrix} + v_t$$

This allows the VAR to capture the information content of the entire panel while remaining parsimonious.

**Key reference:** Bernanke, B.S., Boivin, J. & Eliasz, P. (2005). "Measuring the Effects of Monetary Policy: A Factor-Augmented Vector Autoregressive (FAVAR) Approach." *Quarterly Journal of Economics*, 120(1), 387-422.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

from chronobox import FAVAR

sys.path.insert(0, os.path.join("..", "utils"))
from data_generators import generate_factor_model
from plot_helpers import plot_factors, plot_factor_loadings

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.facecolor"] = "white"
np.set_printoptions(precision=4, suppress=True)

print("All imports loaded successfully.")

## 1. Understanding Factor Models

Before diving into FAVAR, let us understand **factor models**. Given a large panel $X$ of $N$ variables observed over $T$ periods:

$$X_t = \Lambda F_t + e_t$$

where:
- $F_t$ is a $(K \times 1)$ vector of **latent factors** (with $K \ll N$)
- $\Lambda$ is an $(N \times K)$ matrix of **factor loadings**
- $e_t$ is idiosyncratic noise

**PCA estimation** (Stock & Watson, 2002): The factors are estimated as the first $K$ principal components of the standardized data matrix. This is equivalent to solving:

$$\hat{F} = \arg\min_{F, \Lambda} \sum_{t=1}^T \|X_t - \Lambda F_t\|^2$$

subject to the normalization $F'F/T = I_K$.

**Reference:** Stock, J.H. & Watson, M.W. (2002). "Forecasting Using Principal Components from a Large Number of Predictors." *JASA*, 97(460), 1167-1179.

In [ ]:
# Generate synthetic factor model data
# X = Lambda * F + noise, where F follows a VAR(1)
X, F_true, Lambda_true = generate_factor_model(
    n=200, n_series=10, n_factors=2, seed=42
)

print(f"Panel X shape: {X.shape}  (T={X.shape[0]}, N={X.shape[1]})")
print(f"True factors shape: {F_true.shape}  (T, K)")
print(f"True loadings shape: {Lambda_true.shape}  (N, K)")
print(f"\nTrue factor loadings (Lambda):")
series_names = [f"X{i+1}" for i in range(10)]
print(pd.DataFrame(Lambda_true.round(3), index=series_names, columns=["Factor 1", "Factor 2"]))

In [ ]:
# Extract factors via PCA and compare with true factors
from scipy import linalg as la

# Standardize X
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_std[X_std < 1e-10] = 1.0
X_s = (X - X_mean) / X_std

# SVD-based PCA
U, S, Vt = la.svd(X_s, full_matrices=False)
F_pca = U[:, :2] * S[:2]  # first 2 principal components

# Scree plot: how many factors?
explained_var = S**2 / np.sum(S**2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, 11), explained_var[:10] * 100, color="steelblue", alpha=0.8)
axes[0].set_xlabel("Component")
axes[0].set_ylabel("Variance Explained (%)")
axes[0].set_title("Scree Plot: Variance Explained by Each PC")
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=10, color="red", linestyle="--", alpha=0.5, label="10% threshold")
axes[0].legend()

# Cumulative variance
cum_var = np.cumsum(explained_var[:10]) * 100
axes[1].plot(range(1, 11), cum_var, "o-", color="darkred", linewidth=2)
axes[1].set_xlabel("Number of Components")
axes[1].set_ylabel("Cumulative Variance (%)")
axes[1].set_title("Cumulative Variance Explained")
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=90, color="red", linestyle="--", alpha=0.5, label="90% threshold")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "favar_scree_plot.png"), bbox_inches="tight")
plt.show()

print(f"Variance explained by first 2 PCs: {cum_var[1]:.1f}%")
print(f"Variance explained by first 3 PCs: {cum_var[2]:.1f}%")

In [ ]:
# Compare PCA factors with true factors
# Note: PCA factors are identified only up to rotation, so we compare correlations
corr_matrix = np.corrcoef(F_true.T, F_pca.T)[:2, 2:]
print("Correlation between true and PCA factors:")
print(pd.DataFrame(corr_matrix.round(3), 
                    index=["True F1", "True F2"], 
                    columns=["PCA F1", "PCA F2"]))

# Plot true vs estimated factors
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

for i in range(2):
    # Find best matching PCA factor (by absolute correlation)
    best_match = np.argmax(np.abs(corr_matrix[i]))
    sign = np.sign(corr_matrix[i, best_match])
    
    axes[i].plot(F_true[:, i], label=f"True Factor {i+1}", color="steelblue", linewidth=1.5)
    axes[i].plot(sign * F_pca[:, best_match], label=f"PCA Factor {best_match+1} (aligned)", 
                 color="firebrick", linewidth=1.5, linestyle="--")
    axes[i].set_title(f"Factor {i+1}: True vs PCA (corr = {corr_matrix[i, best_match]:.3f})")
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.suptitle("PCA Factor Recovery", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "favar_factor_recovery.png"), bbox_inches="tight")
plt.show()

## 2. Loading US Macro Data for FAVAR

We now work with real-style macroeconomic data. The FAVAR requires:

1. **Panel data** $X_t$: a large set of "informational" variables (e.g., sectoral employment, production indices, financial indicators)
2. **Policy variable** $Y_t$: the variable of primary interest (e.g., the Federal Funds Rate)

In our dataset, we treat `fed_funds` as the policy variable and construct a synthetic panel from the remaining macro variables with transformations.

In [ ]:
# Load US macro data
data_path = os.path.join("..", "data", "us_macro_quarterly.csv")
df = pd.read_csv(data_path, parse_dates=["date"])
df.set_index("date", inplace=True)

print(f"Dataset shape: {df.shape}")
print(f"Variables: {df.columns.tolist()}")
print(f"Period: {df.index[0]} to {df.index[-1]}")
df.describe().round(3)

In [ ]:
# Construct a synthetic panel by creating transformations of base variables
# In practice, FAVAR uses 100+ series (employment by sector, IP by category, etc.)
# Here we create a smaller illustrative panel

np.random.seed(42)
n_obs = len(df)

# Base variables
gdp = df["gdp"].values
inflation = df["inflation"].values
unemp = df["unemployment"].values
ff = df["fed_funds"].values

# Create informational panel (12 series) from base + noise
panel_data = np.column_stack([
    gdp + np.random.normal(0, 0.3, n_obs),                    # GDP proxy 1
    gdp * 0.8 + np.random.normal(0, 0.4, n_obs),              # GDP proxy 2
    gdp * 0.6 + inflation * 0.3 + np.random.normal(0, 0.3, n_obs),  # Mixed 1
    inflation + np.random.normal(0, 0.2, n_obs),               # CPI proxy
    inflation * 0.9 + np.random.normal(0, 0.3, n_obs),         # PPI proxy
    inflation * 0.7 + gdp * 0.2 + np.random.normal(0, 0.25, n_obs), # Mixed 2
    unemp + np.random.normal(0, 0.3, n_obs),                   # Unemp proxy 1
    unemp * 1.1 + np.random.normal(0, 0.4, n_obs),             # Unemp proxy 2
    -gdp * 0.5 + unemp * 0.5 + np.random.normal(0, 0.3, n_obs),  # Output gap proxy
    ff * 0.8 + inflation * 0.2 + np.random.normal(0, 0.3, n_obs),  # Bond yield
    ff * 0.6 + np.random.normal(0, 0.5, n_obs),                # Money market rate
    gdp * 0.3 - unemp * 0.4 + np.random.normal(0, 0.3, n_obs),   # Confidence index
])

panel_names = [
    "IP_total", "IP_manufacturing", "GDP_services",
    "CPI_all", "PPI_finished", "Core_PCE",
    "Unemp_total", "Unemp_duration", "Output_gap",
    "Bond_10y", "Money_mkt", "Confidence"
]

# Policy variable: Federal Funds Rate
policy = ff.reshape(-1, 1)

print(f"Panel shape: {panel_data.shape}  (T, N={len(panel_names)})")
print(f"Policy variable shape: {policy.shape}  (T, M=1)")
print(f"\nPanel variables: {panel_names}")

## 3. Two-Step FAVAR Estimation

The **two-step approach** (Bernanke et al., 2005) proceeds as follows:

**Step 1:** Extract $K$ factors $\hat{F}_t$ from the panel $X$ via PCA. Optionally, remove the component of the factors that is spanned by the policy variable $Y_t$ to ensure the factors capture "slow-moving" real activity rather than monetary policy.

**Step 2:** Estimate a standard VAR on the augmented system $[\hat{F}_t; Y_t]$:

$$\begin{bmatrix} \hat{F}_t \\ Y_t \end{bmatrix} = c + \Phi_1 \begin{bmatrix} \hat{F}_{t-1} \\ Y_{t-1} \end{bmatrix} + \cdots + \Phi_p \begin{bmatrix} \hat{F}_{t-p} \\ Y_{t-p} \end{bmatrix} + v_t$$

The policy shock is identified via Cholesky decomposition, ordering the factors first and the policy variable last (so the factors do not respond contemporaneously to the policy shock).

In [ ]:
# Fit FAVAR with 3 factors, 2 lags, two-step estimation
favar_model = FAVAR(n_factors=3, lags=2, method="two_step")
favar_results = favar_model.fit(panel=panel_data, policy=policy)

print(f"FAVAR estimation complete")
print(f"  Method: {favar_results.method}")
print(f"  Number of factors: {favar_results.n_factors}")
print(f"  Number of policy variables: {favar_results.n_policy}")
print(f"  Panel variables: {favar_results.n_panel}")
print(f"  VAR lags: {favar_results.lags}")
print(f"  Effective observations: {favar_results.n_obs}")
print(f"\nExplained variance by factors:")
for i, ev in enumerate(favar_results.explained_variance_ratio):
    print(f"  Factor {i+1}: {ev*100:.1f}%")
print(f"  Total: {favar_results.total_explained_variance*100:.1f}%")

In [ ]:
# Visualize extracted factors
fig = plot_factors(
    favar_results.factors,
    factor_names=["Factor 1 (Real Activity)", "Factor 2 (Prices)", "Factor 3 (Labor)"],
    dates=df.index,
    title="FAVAR: Extracted Latent Factors",
)
plt.savefig(os.path.join("..", "outputs", "favar_estimated_factors.png"), bbox_inches="tight")
plt.show()

In [ ]:
# Visualize factor loadings as heatmap
fig = plot_factor_loadings(
    favar_results.loadings,
    series_names=panel_names,
    factor_names=["Factor 1", "Factor 2", "Factor 3"],
    title="FAVAR: Factor Loadings (Lambda^f)",
)
plt.savefig(os.path.join("..", "outputs", "favar_loadings_heatmap.png"), bbox_inches="tight")
plt.show()

print("Factor loadings matrix:")
print(pd.DataFrame(
    favar_results.loadings.round(3),
    index=panel_names,
    columns=["Factor 1", "Factor 2", "Factor 3"]
))

## 4. FAVAR Impulse Response Functions

The FAVAR IRF shows the response of the **factor-policy system** $[F_t; Y_t]$ to structural shocks. With Cholesky identification and the policy variable ordered last, the **last shock** is the monetary policy shock.

The IRF in the factor-policy space has shape $(H+1, K+M, K+M)$ where $K$ is the number of factors and $M$ is the number of policy variables.

In [ ]:
# Compute IRFs in the factor-policy space
irf_fy = favar_results.irf(periods=20, identification="cholesky")
print(f"IRF shape (factor-policy space): {irf_fy.shape}")
print(f"  Dimensions: (horizons, {favar_results.n_factors} factors + {favar_results.n_policy} policy, shocks)")

K = favar_results.n_factors
M = favar_results.n_policy
KM = K + M
horizons = np.arange(irf_fy.shape[0])

# The monetary policy shock is the last one (index KM-1)
mp_shock = KM - 1

# Plot response of factors + policy to the monetary policy shock
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
response_names = [f"Factor {i+1}" for i in range(K)] + ["Fed Funds Rate"]
colors = ["steelblue", "firebrick", "darkgreen", "darkorange"]

for i, (ax, name, color) in enumerate(zip(axes.flat, response_names, colors)):
    ax.plot(horizons, irf_fy[:, i, mp_shock], color=color, linewidth=2)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.fill_between(horizons, 0, irf_fy[:, i, mp_shock], alpha=0.15, color=color)
    ax.set_title(f"Response of {name} to Monetary Policy Shock", fontsize=10)
    ax.set_xlabel("Quarters")
    ax.grid(True, alpha=0.3)

plt.suptitle("FAVAR: IRF to Monetary Policy Shock (Factor-Policy Space)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "favar_irf_factors.png"), bbox_inches="tight")
plt.show()

## 5. Panel-Level IRFs

A key advantage of FAVAR is that we can compute IRFs for **every variable** in the panel, not just the few variables in the VAR. The panel-level IRF maps factor-space responses back to observable variables:

$$\text{IRF}_X(h) = \Lambda^f \cdot \Theta_h^{F} + \Lambda^y \cdot \Theta_h^{Y}$$

where $\Theta_h$ are the structural MA coefficients split into factor and policy blocks. This allows us to trace how a monetary policy shock affects *each* of the 12 panel variables.

In [ ]:
# Compute panel-level IRFs (response of all 12 observable series)
irf_panel = favar_results.irf_panel(periods=20, identification="cholesky")
print(f"Panel IRF shape: {irf_panel.shape}")
print(f"  (horizons, N_panel={favar_results.n_panel}, K+M={KM} shocks)")

# Plot response of all panel variables to the monetary policy shock
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
cmap = plt.cm.tab10

for i, (ax, name) in enumerate(zip(axes.flat, panel_names)):
    response = irf_panel[:, i, mp_shock]
    color = cmap(i / 12)
    ax.plot(horizons, response, color=color, linewidth=1.8)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.fill_between(horizons, 0, response, alpha=0.15, color=color)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Quarters")
    ax.grid(True, alpha=0.3)

plt.suptitle("FAVAR: Panel-Level IRF to Monetary Policy Shock", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "favar_irf_panel.png"), bbox_inches="tight")
plt.show()

## 6. Two-Step vs Bayesian Estimation

The FAVAR can also be estimated in a **single step** using Bayesian methods (state-space formulation with Kalman filter). The key difference:

| Feature | Two-Step | Bayesian (One-Step) |
|---------|----------|---------------------|
| Factor extraction | PCA (separate from VAR) | Joint estimation via Kalman filter |
| Uncertainty | Generated regressors problem | Properly accounts for factor uncertainty |
| Speed | Fast | Slower |
| Requirements | None | kalmanbox library |

The two-step approach suffers from a **generated regressors problem**: the factors in Step 2 are treated as known, ignoring estimation uncertainty from Step 1. The Bayesian approach avoids this by jointly estimating factors and VAR parameters.

In [ ]:
# Compare two-step with different number of factors
results_by_k = {}
for k in [1, 2, 3, 4]:
    model_k = FAVAR(n_factors=k, lags=2, method="two_step")
    res_k = model_k.fit(panel=panel_data, policy=policy)
    results_by_k[k] = res_k
    print(f"K={k}: explained variance = {res_k.total_explained_variance*100:.1f}%, "
          f"VAR residual variance (trace) = {np.trace(res_k.var_sigma):.4f}")

# Compare IRFs for different K
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_k = {1: "steelblue", 2: "firebrick", 3: "darkgreen", 4: "darkorange"}

for k, res_k in results_by_k.items():
    irf_k = res_k.irf(periods=20, identification="cholesky")
    km_k = res_k.n_factors + res_k.n_policy
    mp_idx = km_k - 1
    
    # Response of last factor to monetary shock
    axes[0].plot(horizons, irf_k[:, 0, mp_idx], color=colors_k[k], 
                 linewidth=1.5, label=f"K={k}")
    # Response of fed funds to its own shock
    axes[1].plot(horizons, irf_k[:, mp_idx, mp_idx], color=colors_k[k],
                 linewidth=1.5, label=f"K={k}")

axes[0].set_title("Response of Factor 1 to Monetary Shock")
axes[1].set_title("Response of Fed Funds to Own Shock")
for ax in axes:
    ax.axhline(0, color="k", linewidth=0.5)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Quarters")

plt.suptitle("FAVAR: Sensitivity to Number of Factors", fontsize=14)
plt.tight_layout()
plt.show()

## 7. VAR Diagnostics in the FAVAR

We can examine the VAR residuals from the second step to check model adequacy.

In [ ]:
# Examine VAR residuals from the FAVAR
resid = favar_results.var_resid
resid_names = [f"Factor {i+1}" for i in range(K)] + ["Fed Funds"]

print(f"VAR residual shape: {resid.shape}")
print(f"\nResidual covariance matrix:")
print(pd.DataFrame(favar_results.var_sigma.round(4), 
                    index=resid_names, columns=resid_names))

# Residual correlation
corr = np.corrcoef(resid.T)
print(f"\nResidual correlation matrix:")
print(pd.DataFrame(corr.round(3), index=resid_names, columns=resid_names))

# VAR coefficient matrices
print(f"\nVAR coefficient matrices shape: {favar_results.var_coefs.shape}")
print(f"Intercept: {favar_results.intercept.round(4)}")

## Exercicio

### Exercicio 1: Escolha do numero de fatores

O numero de fatores $K$ e uma decisao critica no FAVAR. Bernanke et al. (2005) usam criterios como o de Bai & Ng (2002) para determinar $K$.

1. Estime o FAVAR com $K = 1, 2, 3, 4, 5$ fatores
2. Para cada $K$, compute a variancia explicada total e a IRF do fed_funds ao choque monetario
3. Plote as IRFs sobrepostas — a partir de qual $K$ os resultados se estabilizam?
4. Discuta o trade-off entre parcimonia e capacidade informacional

In [ ]:
# --- Exercicio 1: Seu codigo aqui ---

# Dica:
# for k in range(1, 6):
#     model = FAVAR(n_factors=k, lags=2, method="two_step")
#     res = model.fit(panel=panel_data, policy=policy)
#     irf = res.irf(periods=20, identification="cholesky")
#     # Plot the response of fed_funds to its own shock
#     # irf[:, k, k] is the fed_funds response (last variable)

### Exercicio 2: FAVAR com remocao de Y dos fatores

O parametro `remove_y_from_factors` controla se a componente de $Y_t$ e removida dos fatores estimados. Isso e importante porque, sem essa remocao, os fatores podem "absorver" a variacao da politica monetaria, confundindo a identificacao.

1. Estime dois FAVARs: um com `remove_y_from_factors=True` e outro com `False`
2. Compare os fatores estimados — eles diferem substancialmente?
3. Compare as IRFs do choque monetario nos dois casos
4. Qual abordagem produz IRFs mais plausíveis economicamente?

In [ ]:
# --- Exercicio 2: Seu codigo aqui ---

# Dica:
# favar_with = FAVAR(n_factors=3, lags=2, method="two_step", remove_y_from_factors=True)
# favar_without = FAVAR(n_factors=3, lags=2, method="two_step", remove_y_from_factors=False)
# res_with = favar_with.fit(panel=panel_data, policy=policy)
# res_without = favar_without.fit(panel=panel_data, policy=policy)
# Compare: np.corrcoef(res_with.factors[:, 0], res_without.factors[:, 0])

---

## Resumo

Neste notebook aprendemos:

1. **Modelos de fatores**: como PCA extrai fatores latentes de um painel grande, capturando a co-movimentacao entre centenas de series em poucos fatores

2. **FAVAR (two-step)**: combina extracao de fatores via PCA com estimacao VAR no espaco $[F_t; Y_t]$, permitindo que o VAR "veja" a informacao de todo o painel

3. **IRFs no espaco fatorial e no painel**: podemos computar respostas tanto dos fatores quanto de cada variavel observavel individualmente via `irf_panel()`

4. **Escolha de $K$**: o numero de fatores afeta resultados — criterios como scree plot e Bai-Ng ajudam na decisao

5. **Remocao de $Y$ dos fatores**: passo crucial para que os fatores capturem atividade real e nao politica monetaria

## Referencias

- Bernanke, B.S., Boivin, J. & Eliasz, P. (2005). "Measuring the Effects of Monetary Policy: A FAVAR Approach." *QJE*, 120(1), 387-422.
- Stock, J.H. & Watson, M.W. (2002). "Forecasting Using Principal Components from a Large Number of Predictors." *JASA*, 97(460), 1167-1179.
- Bai, J. & Ng, S. (2002). "Determining the Number of Factors in Approximate Factor Models." *Econometrica*, 70(1), 191-221.

No proximo notebook, veremos o **TVP-VAR** — modelos com parametros que variam no tempo.